# ЛР1 · 03c. Абляция LightGBM: что дают наши дополнительные данные

Вопрос: насколько прогноз улучшают данные, которые мы добавили к самим рядам входов, —
календарь, погода, события, признаки станций? Модель та же, что в `lab-03b` (те же параметры,
тот же период обучения, те же окна теста), меняется только набор признаков. Наборы вложенные:

| Набор | Признаки |
|---|---|
| A | только история самой станции: лаги 24…672 ч, среднее и разброс за 4 прошлые недели, сумма вчерашнего дня |
| B | A + календарь и режимы: час, день недели, праздники, учебные дни, сезон, режимы оплаты |
| C | B + погода (фактическая — допущение «идеального прогноза погоды») |
| D | C + события на площадках и городские события + признаки станции и её номер = полный набор `lab-03b` |

Набор D должен повторить результат `lab-03b` (WAPE 0,0951 на тесте) — это проверка воспроизводимости.
Каждая модель оценивается на тех же 600 окнах теста и на новом срезе (384 станции).

**Kaggle:** Inputs — датасет и вывод `lab-00-setup`; Accelerator — **None** (CPU); Internet не нужен. ~1 ч.

In [ ]:
import glob, json, sys, time, gc
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb
hits = glob.glob("/kaggle/input/**/labkit.py", recursive=True) or glob.glob("labkit.py")
LAB00 = Path(hits[0]).parent
sys.path.insert(0, str(LAB00))
import labkit as lk
print("lightgbm", lgb.__version__)

d = lk.load()
sub = json.load(open(LAB00 / "subset.json", encoding="utf-8"))
stations = sub["stations"]
test_orig = [pd.Timestamp(x) for x in sub["test_origins"]]
val_orig = [pd.Timestamp(x) for x in sub["val_origins"]]
assert stations == lk.make_subset(d)
tmax = d.train_max()
W_test = lk.make_windows(d, stations, test_orig, context=336)
W_val = lk.make_windows(d, stations, val_orig, context=336)

## 1. Признаки (как в lab-03b)

In [ ]:
idx = d.y.index; cols = list(d.y.columns); S = len(cols)
Y = d.y.to_numpy(dtype="float32")
C = d.closed.to_numpy()
Ym = np.where(C, np.nan, Y)
P0 = idx.get_loc(pd.Timestamp("2025-01-01 00:00"))
T = len(idx) - P0

def lag(k):
    return Ym[P0 - k:len(idx) - k]                     # [T, S]

F = {}
for k in [24, 48, 72, 168, 336, 504, 672]:
    F[f"lag{k}"] = lag(k)
F["lag_week_mean4"] = np.nanmean(np.stack([lag(168), lag(336), lag(504), lag(672)]), axis=0)
F["lag_week_std4"] = np.nanstd(np.stack([lag(168), lag(336), lag(504), lag(672)]), axis=0)
# сумма вчерашнего дня (известна в 00:00): для часа h это сумма часов [t-h-24, t-h)
hour = idx.hour.to_numpy()[P0:]
day_sum = pd.DataFrame(Ym).groupby(idx.normalize()).transform("sum").to_numpy()   # сумма своего дня
F["yesterday_total"] = np.vstack([day_sum[P0 - 24:len(idx) - 24]])

# ковариаты по времени: календарь, режимы, события, погода
COV_COLS = [c for c in d.cov.columns if c not in ("dayofyear", "week_of_year")]
cov = d.cov[COV_COLS].to_numpy(dtype="float32")[P0:]

# станции
st = d.static.loc[cols]
ST_COLS = ["scale_weekday", "commute_index", "cluster", "in_cbd", "n_lines", "od_out_flow", "od_in_flow", "latitude", "longitude"]
stv = st[ST_COLS].astype("float32").to_numpy()

# события на площадках
ev = pd.read_parquet(d.root / "events_station_hour.parquet")
ev["station_complex_id"] = ev["station_complex_id"].astype(str)
EV = {}
for c in ["ev_today", "ev_active", "ev_hours_from_start"]:
    wide = ev.pivot_table(index="transit_timestamp", columns="station_complex_id", values=c, aggfunc="first")
    wide = wide.reindex(index=idx, columns=cols)
    EV[c] = wide.to_numpy(dtype="float32")[P0:]
EV["ev_today"] = np.nan_to_num(EV["ev_today"]); EV["ev_active"] = np.nan_to_num(EV["ev_active"])

names = list(F) + list(EV) + COV_COLS + ST_COLS + ["station"]
X = np.empty((T * S, len(names)), dtype="float32")
j = 0
for a in list(F.values()) + list(EV.values()):
    X[:, j] = a.reshape(-1); j += 1
for k in range(cov.shape[1]):
    X[:, j] = np.repeat(cov[:, k], S); j += 1
for k in range(stv.shape[1]):
    X[:, j] = np.tile(stv[:, k], T); j += 1
X[:, j] = np.tile(np.arange(S), T)
y = Y[P0:].reshape(-1)
open_ = ~C[P0:].reshape(-1)
ts = np.repeat(idx[P0:].to_numpy(), S)
print(f"Таблица: {X.shape[0]:,} строк × {X.shape[1]} признаков, {X.nbytes / 1e9:.1f} ГБ")
del F, day_sum; gc.collect()

## 2. Наборы признаков

In [ ]:
WEATHER = {"temperature_2m", "apparent_temperature", "relative_humidity_2m", "precipitation", "rain", "snowfall",
           "snow_depth", "wind_speed_10m", "cloud_cover", "weather_code"}
EVENTS = {"is_city_event", "ev_today", "ev_active", "ev_hours_from_start"}
STATION = set(ST_COLS) | {"station"}
LAGS = [n for n in names if n.startswith("lag") or n == "yesterday_total"]
CAL = [n for n in names if n in COV_COLS and n not in WEATHER and n not in EVENTS]
WEA = [n for n in names if n in WEATHER]
EVS = [n for n in names if n in EVENTS]
STA = [n for n in names if n in STATION]
unassigned = set(names) - set(LAGS + CAL + WEA + EVS + STA)
assert not unassigned, f"признаки без группы: {unassigned}"

def in_order(groups):                 # порядок столбцов как в lab-03b — чтобы набор D повторил её результат точно
    g = set(groups)
    return [n for n in names if n in g]

SETS = {"A_lags": in_order(LAGS),
        "B_lags_calendar": in_order(LAGS + CAL),
        "C_plus_weather": in_order(LAGS + CAL + WEA),
        "D_full": in_order(LAGS + CAL + WEA + EVS + STA)}
assert SETS["D_full"] == names
for k, v in SETS.items():
    print(f"{k:16s} {len(v):2d} признаков")
print("\nкалендарь и режимы:", CAL)
print("погода:", WEA)
print("события:", EVS)
print("станция:", STA)

## 3. Обучение и оценка каждого набора

In [ ]:
tr_end = d.part("train")[1]; va0, va1 = d.part("val")
m_tr = open_ & (ts <= np.datetime64(tr_end))
m_va = open_ & (ts >= np.datetime64(va0)) & (ts <= np.datetime64(va1))
X_tr, y_tr, X_va, y_va = X[m_tr], y[m_tr], X[m_va], y[m_va]
print(f"train {len(y_tr):,} строк, val {len(y_va):,} строк")

params = dict(objective="poisson", learning_rate=0.05, num_leaves=255, min_data_in_leaf=200,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=-1, seed=lk.SEED, verbose=-1)

pos_of = {t: i for i, t in enumerate(idx[P0:])}
col_of = {s: k for k, s in enumerate(cols)}
new_st = [s for s in cols if s not in set(stations)]
W_new = lk.make_windows(d, new_st, test_orig, context=336)

def rows_of(w):
    return np.array([[(pos_of[o] + h) * S + col_of[s] for h in range(lk.HORIZON)]
                     for s, o in zip(w.meta["station"], w.meta["origin"])])
R = {"test": rows_of(W_test), "newslice_test": rows_of(W_new)}
WW = {"test": W_test, "newslice_test": W_new}

summary = []
for key, feats in SETS.items():
    j = [names.index(f) for f in feats]
    cat = [f for f in ["station", "cluster"] if f in feats]
    t0 = time.time()
    dtr = lgb.Dataset(X_tr[:, j], y_tr, feature_name=feats, categorical_feature=cat, free_raw_data=True)
    dva = lgb.Dataset(X_va[:, j], y_va, reference=dtr)
    booster = lgb.train(params, dtr, num_boost_round=1500, valid_sets=[dva], valid_names=["val"],
                        callbacks=[lgb.early_stopping(50, verbose=False)])
    train_min = (time.time() - t0) / 60
    for part, w in WW.items():
        t1 = time.perf_counter()
        pred = booster.predict(X[R[part].reshape(-1)][:, j], num_iteration=booster.best_iteration).reshape(len(w), lk.HORIZON)
        ms = (time.perf_counter() - t1) * 1000 / len(w)
        pred = np.clip(pred, 0, None)
        valid = lk.check_valid(list(pred), w, tmax)
        point = lk.apply_fallback(list(pred), valid, w)
        row = lk.save_run("lightgbm_abl", f"{key}_{part}", w, point, valid, None,
                          {"ms_p50": ms, "ms_p95": ms, "total_s": train_min * 60},
                          {"n_features": len(feats), "n_trees": booster.best_iteration,
                           "train_minutes": round(train_min, 1)})
        summary.append({"набор": key, "часть": part, "признаков": len(feats), "WAPE": row["wape"],
                        "ДИ низ": row["wape_ci_lo"], "ДИ верх": row["wape_ci_hi"], "MASE": row["mase"],
                        "деревьев": booster.best_iteration, "мин": round(train_min, 1)})
    print(f"{key:16s} test WAPE={summary[-2]['WAPE']:.4f}  newslice WAPE={summary[-1]['WAPE']:.4f}  "
          f"деревьев {booster.best_iteration}, {train_min:.1f} мин")
    del dtr, dva, booster; gc.collect()

## 4. Итог и значимость

Парный бутстрэп на одних и тех же окнах: насколько каждый следующий набор лучше предыдущего.

In [ ]:
res = pd.DataFrame(summary)
print(res.round(4).to_string(index=False))
res.to_csv(lk.results_dir() / "ablation_summary.csv", index=False)

def ws(key, part):
    return pd.read_parquet(lk.results_dir() / f"lightgbm_abl__{key}_{part}__windows.parquet")

keys = list(SETS)
rows = []
for part in WW:
    for a, b_ in zip(keys[:-1], keys[1:]):
        diff, lo, hi = lk.paired_diff_ci(ws(b_, part), ws(a, part), "wape")
        rows.append({"часть": part, "сравнение": f"{b_} − {a}", "разность WAPE": diff, "ДИ низ": lo, "ДИ верх": hi,
                     "значимо": "да" if lo > 0 or hi < 0 else "нет"})
steps = pd.DataFrame(rows)
print("\nВклад каждой группы признаков (отрицательная разность = стало лучше):")
print(steps.round(4).to_string(index=False))
steps.to_csv(lk.results_dir() / "ablation_steps.csv", index=False)